# 04 — Data Drift Detection
Apply KS Test + PSI for numerical features and Chi-Square Test for categorical features.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from preprocessing import preprocess, get_feature_types
from drift import detect_numerical_drift, detect_categorical_drift, run_drift_detection
from report import build_report, save_report_to_disk

## 1. Load & Preprocess

In [2]:
ref_df = preprocess(pd.read_csv('../data/reference.csv'))
cur_df = preprocess(pd.read_csv('../data/current.csv'))
ft = get_feature_types(ref_df)
num_cols = ft['numerical']
cat_cols = ft['categorical']
print('Features —', 'Numerical:', len(num_cols), '| Categorical:', len(cat_cols))

Features — Numerical: 6 | Categorical: 2


/Users/animeshsingh/Downloads/ML-Data-Drift-Dashboard/notebooks/../src/preprocessing.py:63: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(sample, infer_datetime_format=True, errors="coerce")
/Users/animeshsingh/Downloads/ML-Data-Drift-Dashboard/notebooks/../src/preprocessing.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, infer_datetime_format=True, errors="coerce")
/Users/animeshsingh/Downloads/ML-Data-Drift-Dashboard/notebooks/../src/preprocessing.py:63: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict versi

## 2. Numerical Drift (KS Test + PSI)

In [3]:
num_drift = detect_numerical_drift(ref_df, cur_df, num_cols)
display(num_drift)

,feature,feature_type,method,statistic,p_value,psi,drift_status,psi_status
0,age,Numerical,KS Test,0.225,0.265687,1.223540,No Drift,Significant Drift
1,income,Numerical,KS Test,0.175,0.578600,0.226604,No Drift,Significant Drift
2,credit_score,Numerical,KS Test,0.100,0.990019,0.045815,No Drift,No Drift
3,loan_amount,Numerical,KS Test,0.125,0.918805,0.197215,No Drift,Moderate Drift
4,employment_years,Numerical,KS Test,0.175,0.578600,1.392906,No Drift,Significant Drift
5,default,Numerical,KS Test,0.200,0.404587,0.000000,No Drift,No Drift


## 3. Categorical Drift (Chi-Square Test)

In [4]:
cat_drift = detect_categorical_drift(ref_df, cur_df, cat_cols)
display(cat_drift)

,feature,feature_type,method,statistic,p_value,psi,drift_status,psi_status
0,education,Categorical,Chi-Square Test,0.0,1.0,None,No Drift,None
1,loan_purpose,Categorical,Chi-Square Test,0.0,1.0,None,No Drift,None


## 4. Combined Drift Report

In [5]:
drift_df = run_drift_detection(ref_df, cur_df, num_cols, cat_cols)
report = build_report(drift_df)
display(report)

/Users/animeshsingh/Downloads/ML-Data-Drift-Dashboard/notebooks/../src/drift.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([num_drift, cat_drift], ignore_index=True)


,Feature,Feature Type,Method,Statistic,P Value,Psi,Drift Status,Psi Status
0,age,Numerical,KS Test,0.225,0.265687,1.223540,No Drift,Significant Drift
1,income,Numerical,KS Test,0.175,0.578600,0.226604,No Drift,Significant Drift
2,credit_score,Numerical,KS Test,0.100,0.990019,0.045815,No Drift,No Drift
3,loan_amount,Numerical,KS Test,0.125,0.918805,0.197215,No Drift,Moderate Drift
4,employment_years,Numerical,KS Test,0.175,0.578600,1.392906,No Drift,Significant Drift
5,default,Numerical,KS Test,0.200,0.404587,0.000000,No Drift,No Drift
6,education,Categorical,Chi-Square Test,0.000,1.000000,NaN,No Drift,None
7,loan_purpose,Categorical,Chi-Square Test,0.000,1.000000,NaN,No Drift,None


## 5. Drift Summary

In [6]:
total = len(drift_df)
drifted = drift_df['drift_status'].str.contains('Drift', na=False).sum()
print(f'Total features tested : {total}')
print(f'Drift detected        : {drifted} ({round(drifted/total*100, 1)}%)')
print(f'Stable features       : {total - drifted}')

Total features tested : 8
Drift detected        : 8 (100.0%)
Stable features       : 0


## 6. Save Report to Disk

In [7]:
import os
os.makedirs('../reports', exist_ok=True)
csv_path, xl_path = save_report_to_disk(
    report,
    csv_path='../reports/drift_report.csv',
    excel_path='../reports/drift_report.xlsx'
)
print('Saved:', csv_path, xl_path)

Saved: ../reports/drift_report.csv ../reports/drift_report.xlsx
